# TRChat-50M CPT + SFT Eğitimi

Bu notebook, TRChat-50M modelini Google Colab üzerinde sürekli ön-eğitim (CPT) ve denetimli ince ayar (SFT) ile eğitmek için hazırlanmıştır.

Kullanım:
1. Bu klasörü (`colab_export`) Google Drive'ınızdaki `my-50m-model-colab` klasörüne yükleyin.
2. Notebooku Colab'de açın ve GPU çalışma zamanı seçin (T4 önerilir).
3. Hücreleri sırayla çalıştırın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Ortam hazırlığı

In [ ]:
import os
PROJECT_DIR = '/content/drive/MyDrive/my-50m-model-colab'
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
!pip install -q -r requirements.txt

## 1. CPT (Continued Pre-Training)

In [ ]:
# CPT'yi başlat. İstersen --max-steps ve --micro-batch ile sınırlandırabilirsin.
!python scripts/train_cpt.py --config configs/training.yaml

## 2. SFT (Supervised Fine-Tuning)

In [ ]:
# SFT, checkpoints/best_cpt veya en son checkpoint üzerinden devam eder.
!python scripts/train_sft.py --config configs/training.yaml

## 3. Hızlı test

In [ ]:
from transformers import AutoTokenizer, LlamaForCausalLM
import torch

model_dir = 'models/my-model-final'
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = LlamaForCausalLM.from_pretrained(model_dir).to('cuda' if torch.cuda.is_available() else 'cpu')

prompt = "Türkiye'nin başkenti "
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## GGUF Export (llama.cpp için)

In [ ]:
import subprocess
from pathlib import Path

final_model = Path('models/trchat-50m-final')
if not final_model.exists():
    print('[GGUF] Final model not found. Run SFT first.')
else:
    print('[GGUF] Starting GGUF export...')
    subprocess.run([
        'python', 'scripts/export_gguf.py',
        '--model-dir', 'models/trchat-50m-final',
        '--out-dir', 'models/gguf',
        '--quantizations', 'fp16,q8_0,q4_k_m,q5_k_m'
    ], check=True)
    print('[GGUF] Done. Files saved to models/gguf/')